In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_ROOT = os.environ.get("DATA_ROOT", "./data/wildlife_dataset")

TRAIN_DIR = os.path.join(DATA_ROOT, "datawildlife_train")
VAL_DIR   = os.path.join(DATA_ROOT, "datawildlife_val")
TEST_DIR  = os.path.join(DATA_ROOT, "datawildlife_test")

In [ ]:
%cd /content/MotionTrack
!pip install -r requirements.txt -q
!pip install cython_bbox -q

In [ ]:
import os, re, json, shutil

def get_num(fname):
    m = re.search(r'(\d+)', fname)
    return int(m.group(1)) if m else None

data_roots = [VAL_DIR, TEST_DIR]
# CLASS_MAP = {"elephant": 1, "zebra": 2, "giraffe": 3} # used for tracking by origin weight, which is not finetune
CLASS_MAP = {"elephant": 20, "zebra": 22, "giraffe": 23} # used for tracking by finetune weight

for data_root in data_roots:
    for seq in sorted(os.listdir(data_root)):
        seq_path = os.path.join(data_root, seq)
        frames_path = os.path.join(seq_path, 'frames')
        boxid_path = os.path.join(seq_path, 'boxid')
        if not os.path.isdir(frames_path) or not os.path.isdir(boxid_path):
            continue

        # 1. Sort ảnh THEO SỐ GỐC trong tên file (numeric, không phải string)
        img_files = [f for f in os.listdir(frames_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        img_files_sorted = sorted(img_files, key=get_num)  # đảm bảo đúng thứ tự thời gian thật

        # 2. Map: số gốc trong tên ảnh -> rank (1-indexed) sau khi sort
        orig_num_to_rank = {get_num(f): i+1 for i, f in enumerate(img_files_sorted)}

        # 3. Rename ảnh theo RANK (không theo số gốc), qua thư mục tạm
        tmp = frames_path + "_tmp"
        os.makedirs(tmp, exist_ok=True)
        ext = os.path.splitext(img_files_sorted[0])[1].lower()
        for i, f in enumerate(img_files_sorted):
            shutil.move(os.path.join(frames_path, f), os.path.join(tmp, f"{i+1:06d}{ext}"))
        for f in os.listdir(tmp):
            shutil.move(os.path.join(tmp, f), os.path.join(frames_path, f))
        os.rmdir(tmp)

        # 4. Tạo gt.txt dùng ĐÚNG rank vừa map, không dùng +1 cứng nữa
        gt_dir = os.path.join(seq_path, 'gt')
        os.makedirs(gt_dir, exist_ok=True)
        rows = []
        for fname in os.listdir(boxid_path):
            if not fname.endswith('.json'):
                continue
            orig_num = get_num(fname)
            if orig_num is None or orig_num not in orig_num_to_rank:
                print(f"  ⚠️ {seq}: {fname} không khớp ảnh nào trong frames/, bỏ qua")
                continue
            frame_id = orig_num_to_rank[orig_num]  # dùng rank, không dùng orig_num+1

            with open(os.path.join(boxid_path, fname)) as f:
                data = json.load(f)
            for shape in data.get("shapes", []):
                pts = shape["points"]
                x1, y1 = min(p[0] for p in pts), min(p[1] for p in pts)
                x2, y2 = max(p[0] for p in pts), max(p[1] for p in pts)
                w, h = x2 - x1, y2 - y1
                track_id = shape.get("group_id")
                if track_id is None:
                    continue
                cls = CLASS_MAP.get(shape.get("label", ""), 1)
                rows.append((frame_id, int(track_id)+1, x1, y1, w, h, 1, cls, 1))

        rows.sort(key=lambda r: (r[0], r[1]))
        with open(os.path.join(gt_dir, 'gt.txt'), 'w') as f:
            for r in rows:
                f.write(f"{r[0]},{r[1]},{r[2]:.2f},{r[3]:.2f},{r[4]:.2f},{r[5]:.2f},{r[6]},{r[7]},{r[8]}\n")

        print(f"✓ {seq}: {len(img_files_sorted)} frames renamed, {len(rows)} gt annotations")

In [ ]:
import cv2
import configparser
import os

for split_name, data_root in [
    ("val", VAL_DIR),
    ("test", TEST_DIR),
]:
    if not os.path.isdir(data_root):
        continue

    print(f"\n📂 Đang xử lý: {split_name} ({data_root})")
    seqs = sorted(os.listdir(data_root))

    for seq in seqs:
        seq_path = os.path.join(data_root, seq)
        if not os.path.isdir(seq_path):
            continue

        ini_path = os.path.join(seq_path, 'seqinfo.ini')
        frames_path = os.path.join(seq_path, 'frames')

        if not os.path.exists(frames_path):
            print(f"  ❌ {seq}: không có folder frames/")
            continue

        imgs = sorted([f for f in os.listdir(frames_path) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        if not imgs:
            print(f"  ❌ {seq}: folder frames/ rỗng")
            continue

        img = cv2.imread(os.path.join(frames_path, imgs[0]))
        h, w = img.shape[:2]

        cfg = configparser.ConfigParser()
        cfg['Sequence'] = {
            'name': seq,
            'imDir': 'frames',
            'frameRate': '30',
            'seqLength': str(len(imgs)),
            'imWidth': str(w),
            'imHeight': str(h),
            'imExt': os.path.splitext(imgs[0])[1]
        }
        with open(ini_path, 'w') as f:
            cfg.write(f)
        print(f"  ✅ {seq}: {w}x{h}, {len(imgs)} frames")

In [ ]:
import os
import re

track_path = "/content/MotionTrack/tools/track.py"
with open(track_path, "r") as f:
    content = f.read()

# --- Patch 1: read seqs from data_root instead of hardcoded list ---
content = content.replace(
    "seqs_str = '2,18,22,28,32,41,45,46,55,62,69,80,87,89,90,101,102,103,105'\n    seqs = [seq.strip() for seq in seqs_str.split(',')]",
    "seqs = sorted([s for s in os.listdir(data_root) if os.path.isdir(os.path.join(data_root, s))])"
)

# --- Patch 2: fix image path (data_root/seq/seq -> data_root/seq/frames) ---
content = content.replace(
    "dataloader = LoadPaths(os.path.join(data_root, seq, seq))",
    "dataloader = LoadPaths(os.path.join(data_root, seq, 'frames'))"
)

# --- Patch 3: print full traceback on failure ---
if "import traceback" not in content:
    content = content.replace("import argparse", "import argparse\nimport traceback")

content = content.replace(
    "    track(opt, opt.weights, opt.batch_size, save_txt=opt.save_txt |\n          opt.save_hybrid, trace=not opt.no_trace)",
    "    try:\n        track(opt, opt.weights, opt.batch_size, save_txt=opt.save_txt |\n              opt.save_hybrid, trace=not opt.no_trace)\n    except Exception as e:\n        traceback.print_exc()"
)

# --- Patch 4: skip eval for seqs without gt.txt ---
old1 = "    accs = []\n    n_frame = 0"
new1 = "    accs = []\n    seqs_for_eval = []\n    n_frame = 0"
assert old1 in content, "❌ Pattern 'accs = []' not found — check file content"
content = content.replace(old1, new1)

old2 = "        # eval\n        logger.info('Evaluate seq: {}'.format(seq))\n        evaluator = Evaluator(data_root, seq, data_type)\n        accs.append(evaluator.eval_file(result_filename))"
new2 = """        # eval only if gt.txt exists
        gt_path = os.path.join(data_root, seq, 'gt', 'gt.txt')
        if os.path.exists(gt_path):
            logger.info('Evaluate seq: {}'.format(seq))
            evaluator = Evaluator(data_root, seq, data_type)
            accs.append(evaluator.eval_file(result_filename))
            seqs_for_eval.append(seq)
        else:
            logger.info('No gt.txt for seq: {}, skipping eval'.format(seq))"""
assert old2 in content, "❌ Pattern 'eval block' not found — file may already be patched"
content = content.replace(old2, new2)

# --- Patch 5: compute metrics only over seqs that had gt.txt ---
old3 = "    summary = Evaluator.get_summary(accs, seqs, metrics)\n    strsummary = mm.io.render_summary(\n        summary, formatters=mh.formatters, namemap=mm.io.motchallenge_metric_names)\n    print(strsummary)\n    logger.info(strsummary)"
new3 = """    if len(accs) > 0:
        summary = Evaluator.get_summary(accs, seqs_for_eval, metrics)
        strsummary = mm.io.render_summary(
            summary, formatters=mh.formatters, namemap=mm.io.motchallenge_metric_names)
        print(strsummary)
        logger.info(strsummary)
    else:
        logger.info('No sequences with gt.txt found, skipping metric computation')"""
assert old3 in content, "❌ Pattern 'summary block' not found — file may already be patched"
content = content.replace(old3, new3)

with open(track_path, "w") as f:
    f.write(content)

In [ ]:
#Điều kiện để weights load đúng
with open('/content/MotionTrack/utils/google_utils.py', 'r') as f:
    lines = f.readlines()

lines.insert(19, '    if os.path.isfile(file):\n        return\n')

with open('/content/MotionTrack/utils/google_utils.py', 'w') as f:
    f.writelines(lines)


In [ ]:
with open('/content/MotionTrack/utils/google_utils.py', 'r') as f:
    lines = f.readlines()

lines[19] = '    if os.path.isfile(file):\n        return\n'

with open('/content/MotionTrack/utils/google_utils.py', 'w') as f:
    f.writelines(lines)

In [ ]:
with open('/content/MotionTrack/utils/google_utils.py', 'r') as f:
    lines = f.readlines()

lines[19] = '    if os.path.isfile(file):\n'
lines.insert(20, '        return\n')

if lines[21].strip() == 'return':
    lines.pop(21)

with open('/content/MotionTrack/utils/google_utils.py', 'w') as f:
    f.writelines(lines)

In [ ]:
import motmetrics
import os

dist_file = os.path.join(os.path.dirname(motmetrics.__file__), "distances.py")

with open(dist_file) as f:
    content = f.read()

if "np.asfarray" in content:
    content = content.replace("np.asfarray(", "np.asarray(")
    with open(dist_file, "w") as f:
        f.write(content)

In [ ]:
with open('/content/MotionTrack/tools/track.py', 'r') as f:
    content = f.read()

# 1. Bật lưu ảnh visualization
content = content.replace(
    "    save_images = False\n    save_videos = False",
    "    save_images = True\n    save_videos = True"
)

# 2. Thêm hàm vẽ box + ID + confidence score (chèn trước eval_seq)
draw_func = '''
def draw_tracking_result(img, tlwhs, obj_ids, scores, class_ids, class_names, frame_id=0, fps=0.):
    im = np.ascontiguousarray(np.copy(img))
    text_scale = 1.5
    text_thickness = 2
    line_thickness = 2

    cv2.putText(im, 'frame: %d fps: %.2f num: %d' % (frame_id, fps, len(tlwhs)),
                (0, int(15 * text_scale)), cv2.FONT_HERSHEY_PLAIN, text_scale,
                (0, 0, 255), thickness=2)

    for i, tlwh in enumerate(tlwhs):
        x1, y1, w, h = tlwh
        intbox = tuple(map(int, (x1, y1, x1 + w, y1 + h)))
        obj_id = int(obj_ids[i])
        score = scores[i]
        cls_id = int(class_ids[i])
        cls_name = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)

        color = (int(37 * obj_id % 255), int(17 * obj_id % 255), int(29 * obj_id % 255))
        cv2.rectangle(im, intbox[0:2], intbox[2:4], color=color, thickness=line_thickness)

        label = 'ID:{} {} {:.2f}'.format(obj_id, cls_name, score)
        cv2.putText(im, label, (intbox[0], max(0, intbox[1] - 5)), cv2.FONT_HERSHEY_PLAIN,
                    text_scale, (0, 255, 255), thickness=text_thickness)
    return im


'''
content = content.replace(
    "def eval_seq(args, exp, predictor, model, dataloader, data_type, result_filename, save_dir=None, show_image=False, frame_rate=30,):",
    draw_func + "def eval_seq(args, exp, predictor, model, dataloader, data_type, result_filename, save_dir=None, show_image=False, frame_rate=30,):"
)

# 3. Thay lời gọi plot_tracking_with_class bằng draw_tracking_result (có truyền score)
old_call = """        if show_image or save_dir is not None:
            # online_im = plot_tracking(img_info['raw_img'],online_tlwhs,online_ids,frame_id=frame_id + 1,fps=1.0 / timer.average_time)
            online_im = plot_tracking_with_class(
                img_info['raw_img'], online_tlwhs, online_ids, online_classids, args.class_names, frame_id=frame_id + 1, fps=1.0 / timer.average_time)"""

new_call = """        if show_image or save_dir is not None:
            online_im = draw_tracking_result(
                img_info['raw_img'], online_tlwhs, online_ids, online_scores, online_classids,
                args.class_names, frame_id=frame_id + 1, fps=1.0 / timer.average_time)"""

content = content.replace(old_call, new_call)

with open('/content/MotionTrack/tools/track.py', 'w') as f:
    f.write(content)

In [ ]:
with open('/content/MotionTrack/models/experimental.py', 'r') as f:
    content = f.read()

content = content.replace(
    'ckpt = torch.load(w, map_location=map_location)  # load',
    'ckpt = torch.load(w, map_location=map_location, weights_only=False)  # load'
)

with open('/content/MotionTrack/models/experimental.py', 'w') as f:
    f.write(content)

In [ ]:
with open('/content/MotionTrack/tracker/matching.py', 'r') as f:
    content = f.read()

content = content.replace('dtype=np.float)', 'dtype=np.float64)')

with open('/content/MotionTrack/tracker/matching.py', 'w') as f:
    f.write(content)

In [ ]:
for fpath in [
    '/content/MotionTrack/tracker/motion_tracker.py',
    '/content/MotionTrack/tracker/nokalman_tracker.py'
]:
    with open(fpath, 'r') as f:
        content = f.read()
    content = content.replace('dtype=np.float)', 'dtype=np.float64)')
    with open(fpath, 'w') as f:
        f.write(content)

In [ ]:
path = '/content/MotionTrack/tools/track.py'

with open(path, 'r') as f:
    content = f.read()

old = '''        if self.device == "gpu":
            img = img.cuda()
            if self.fp16:
                img = img.half()  # to FP16'''

new = '''        img = img.to(self.device)
        if self.fp16:
            img = img.half()  # to FP16'''

content = content.replace(old, new)

with open(path, 'w') as f:
    f.write(content)

In [ ]:
path = '/content/MotionTrack/tools/track.py'

with open(path, 'r') as f:
    content = f.read()

old = '''predictor = Predictor(model, exp, device="gpu",fp16=opt.fp16, reid=opt.reid)'''
new = '''predictor = Predictor(model, exp, device=device, fp16=opt.fp16, reid=opt.reid)'''

content = content.replace(old, new)

with open(path, 'w') as f:
    f.write(content)

Đã patch xong: Predictor dùng đúng torch.device thay vì chuỗi 'gpu'


In [ ]:
%cd /content/MotionTrack

!python tools/track.py \
    --weights /content/drive/MyDrive/yolov7_weight/yolov7-w6.pt\
    --data data/coco.yaml \
    --num_classes 80 \
    --class_names "person,bicycle,car,motorcycle,airplane,bus,train,truck,boat,traffic light,fire hydrant,stop sign,parking meter,bench,bird,cat,dog,horse,sheep,cow,elephant,bear,zebra,giraffe,backpack,umbrella,handbag,tie,suitcase,frisbee,skis,snowboard,sports ball,kite,baseball bat,baseball glove,skateboard,surfboard,tennis racket,bottle,wine glass,cup,fork,knife,spoon,bowl,banana,apple,sandwich,orange,broccoli,carrot,hot dog,pizza,donut,cake,chair,couch,potted plant,bed,dining table,toilet,tv,laptop,mouse,remote,keyboard,cell phone,microwave,oven,toaster,sink,refrigerator,book,clock,vase,scissors,teddy bear,hair drier,toothbrush" \
    --data_root /content/wildlife/datawildlife_val_track \
    --device 0 \
    --project /content/drive/MyDrive/runs_last/track_yolo_w6 \
    --name wildlife_exp \
    --no-trace \
    --exist-ok

/content/MotionTrack
Namespace(weights=['/content/drive/MyDrive/yolov7_weight/yolov7-w6.pt'], data='data/coco.yaml', batch_size=1, test_size=[1088, 1920], conf_thres=0.1, iou_thres=0.65, device='0', augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, project='/content/drive/MyDrive/runs_last/track_yolo_w6', name='wildlife_exp', exist_ok=True, no_trace=True, num_classes=80, class_names='person,bicycle,car,motorcycle,airplane,bus,train,truck,boat,traffic light,fire hydrant,stop sign,parking meter,bench,bird,cat,dog,horse,sheep,cow,elephant,bear,zebra,giraffe,backpack,umbrella,handbag,tie,suitcase,frisbee,skis,snowboard,sports ball,kite,baseball bat,baseball glove,skateboard,surfboard,tennis racket,bottle,wine glass,cup,fork,knife,spoon,bowl,banana,apple,sandwich,orange,broccoli,carrot,hot dog,pizza,donut,cake,chair,couch,potted plant,bed,dining table,toilet,tv,laptop,mouse,remote,keyboard,cell phone,microwave,oven,toaster,sink,refrigerator,book,clock,vase,sc

In [ ]:
%cd /content/MotionTrack

!python tools/track.py \
    --weights /content/drive/MyDrive/yolov7_weight/yolov7-tiny.pt\
    --data data/coco.yaml \
    --num_classes 80 \
    --class_names "person,bicycle,car,motorcycle,airplane,bus,train,truck,boat,traffic light,fire hydrant,stop sign,parking meter,bench,bird,cat,dog,horse,sheep,cow,elephant,bear,zebra,giraffe,backpack,umbrella,handbag,tie,suitcase,frisbee,skis,snowboard,sports ball,kite,baseball bat,baseball glove,skateboard,surfboard,tennis racket,bottle,wine glass,cup,fork,knife,spoon,bowl,banana,apple,sandwich,orange,broccoli,carrot,hot dog,pizza,donut,cake,chair,couch,potted plant,bed,dining table,toilet,tv,laptop,mouse,remote,keyboard,cell phone,microwave,oven,toaster,sink,refrigerator,book,clock,vase,scissors,teddy bear,hair drier,toothbrush" \
    --data_root /content/wildlife/datawildlife_val_track \
    --device 0 \
    --project /content/drive/MyDrive/runs_last/track_yolo_tiny \
    --name wildlife_exp \
    --no-trace \
    --exist-ok

/content/MotionTrack
Namespace(weights=['/content/drive/MyDrive/yolov7_weight/yolov7-tiny.pt'], data='data/coco.yaml', batch_size=1, test_size=[1088, 1920], conf_thres=0.1, iou_thres=0.65, device='0', augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, project='/content/drive/MyDrive/runs_last/track_yolo_tiny', name='wildlife_exp', exist_ok=True, no_trace=True, num_classes=80, class_names='person,bicycle,car,motorcycle,airplane,bus,train,truck,boat,traffic light,fire hydrant,stop sign,parking meter,bench,bird,cat,dog,horse,sheep,cow,elephant,bear,zebra,giraffe,backpack,umbrella,handbag,tie,suitcase,frisbee,skis,snowboard,sports ball,kite,baseball bat,baseball glove,skateboard,surfboard,tennis racket,bottle,wine glass,cup,fork,knife,spoon,bowl,banana,apple,sandwich,orange,broccoli,carrot,hot dog,pizza,donut,cake,chair,couch,potted plant,bed,dining table,toilet,tv,laptop,mouse,remote,keyboard,cell phone,microwave,oven,toaster,sink,refrigerator,book,clock,vas

In [ ]:
%cd /content/MotionTrack

!python tools/track.py \
    --weights /content/drive/MyDrive/yolov7_weight/yolov7.pt\
    --data data/coco.yaml \
    --num_classes 80 \
    --class_names "person,bicycle,car,motorcycle,airplane,bus,train,truck,boat,traffic light,fire hydrant,stop sign,parking meter,bench,bird,cat,dog,horse,sheep,cow,elephant,bear,zebra,giraffe,backpack,umbrella,handbag,tie,suitcase,frisbee,skis,snowboard,sports ball,kite,baseball bat,baseball glove,skateboard,surfboard,tennis racket,bottle,wine glass,cup,fork,knife,spoon,bowl,banana,apple,sandwich,orange,broccoli,carrot,hot dog,pizza,donut,cake,chair,couch,potted plant,bed,dining table,toilet,tv,laptop,mouse,remote,keyboard,cell phone,microwave,oven,toaster,sink,refrigerator,book,clock,vase,scissors,teddy bear,hair drier,toothbrush" \
    --data_root /content/wildlife/datawildlife_val_track \
    --device 0 \
    --project /content/drive/MyDrive/runs_last/track_yolo_normal \
    --name wildlife_exp \
    --no-trace \
    --exist-ok

/content/MotionTrack
Namespace(weights=['/content/drive/MyDrive/yolov7_weight/yolov7.pt'], data='data/coco.yaml', batch_size=1, test_size=[1088, 1920], conf_thres=0.1, iou_thres=0.65, device='0', augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, project='/content/drive/MyDrive/runs_last/track_yolo_normal', name='wildlife_exp', exist_ok=True, no_trace=True, num_classes=80, class_names='person,bicycle,car,motorcycle,airplane,bus,train,truck,boat,traffic light,fire hydrant,stop sign,parking meter,bench,bird,cat,dog,horse,sheep,cow,elephant,bear,zebra,giraffe,backpack,umbrella,handbag,tie,suitcase,frisbee,skis,snowboard,sports ball,kite,baseball bat,baseball glove,skateboard,surfboard,tennis racket,bottle,wine glass,cup,fork,knife,spoon,bowl,banana,apple,sandwich,orange,broccoli,carrot,hot dog,pizza,donut,cake,chair,couch,potted plant,bed,dining table,toilet,tv,laptop,mouse,remote,keyboard,cell phone,microwave,oven,toaster,sink,refrigerator,book,clock,vase,s

In [ ]:
%cd /content/MotionTrack

!python tools/track.py \
    --weights /content/drive/MyDrive/yolov7_weight/last_yolov7_tiny.pt\
    --data data/wildlife.yaml \
    --num_classes 3 \
    --class_names "elephant,zebra,giraffe" \
    --data_root /content/wildlife/datawildlife_val_track \
    --device 0 \
    --project /content/drive/MyDrive/runs_last/track_yolo_tiny_finetune \
    --name wildlife_exp \
    --no-trace \
    --exist-ok

/content/MotionTrack
Namespace(weights=['/content/drive/MyDrive/yolov7_weight/last_yolov7_tiny.pt'], data='data/wildlife.yaml', batch_size=1, test_size=[1088, 1920], conf_thres=0.1, iou_thres=0.65, device='0', augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, project='/content/drive/MyDrive/runs_last/track_yolo_tiny_finetune', name='wildlife_exp', exist_ok=True, no_trace=True, num_classes=3, class_names='elephant,zebra,giraffe', first_track_thresh=0.6, second_track_thresh=0.1, det_thresh=0.7, first_match_thresh=0.98, second_match_thresh=0.98, motion_match_thresh=0.98, unconfirmed_match_thresh_iou=0.98, unconfirmed_match_thresh_motion=0.98, track_buffer=30, motion_thresh=140, use_motion=True, data_root='/content/wildlife/datawildlife_val_track', min_box_area=0, reid=False, fp16=True)
2026-08-05 09:17:51 | INFO     | __main__:229 - Args: Namespace(weights=['/content/drive/MyDrive/yolov7_weight/last_yolov7_tiny.pt'], data='data/wildlife.yaml', batch_size=1,

In [ ]:
%cd /content/MotionTrack

!python tools/track.py \
    --weights /content/drive/MyDrive/yolov7_weight/last_yolov7_w6.pt\
    --data data/wildlife.yaml \
    --num_classes 3 \
    --class_names "elephant,zebra,giraffe" \
    --data_root /content/wildlife/datawildlife_val_track \
    --device 0 \
    --project /content/drive/MyDrive/runs_last/track_yolo_w6_finetune \
    --name wildlife_exp \
    --no-trace \
    --exist-ok